# Hit Tuning studies

We're using the latest version of `icaruscode`, `v10_06_00_06p03` with all the features included from the work Matteo Vicenzi has been doing on the PMT-overlay and some of the fixes introduced by Micheal Carrigan on the Hit Finder settings



In [8]:
import uproot
import mplhep as hep

# import ultraplot as plot
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import pandas as pd
import xarray as xr

import sqlite3 as sql
import corner

sns.reset_orig()

In [ ]:
connection = sql.connect('hitTuning_merged_test.db')
data = pd.read_sql('SELECT * FROM runs', connection)

In [ ]:
data.keys()

In [ ]:
data_electronOnly = data[['ratio_ele0', 'ratio_ele1', 'ratio_ele2']]

data_max = data_electronOnly.sort_values(['ratio_ele0', 'ratio_ele1', 'ratio_ele2'])
max_ele0, max_ele1, max_ele2 = data_max.max()

max_common_ele0, max_common_ele1, max_common_ele2 = data_max.iloc[0]

figure = corner.corner(data_electronOnly.to_numpy(), plot_contours=False, plot_density=False)
corner.overplot_lines(figure, [max_ele0, max_ele1, max_ele2], lw=5)
corner.overplot_lines(figure, [max_common_ele0, max_common_ele1, max_common_ele2], lw=5, color='r')


In [ ]:
data[['ratio_ele', 'ratio_ele0', 'ratio_ele1', 'ratio_ele2']]

In [ ]:

data_max

In [ ]:
data_electronOnly = data[['ratio_p0', 'ratio_p1', 'ratio_p2']]

data_max = data_electronOnly.sort_values(['ratio_p0', 'ratio_p2', 'ratio_p1', ])
max_0, max_1, max_2 = data_max.max()

max_common_0, max_common_1, max_common_2 = data_max.iloc[0]

figure = corner.corner(data_electronOnly.to_numpy())
# corner.overplot_lines(figure, [max_0, max_1, max_2 ], lw=5, alpha=0.5)
corner.overplot_lines(figure, [max_common_0, max_common_1, max_common_2], lw=5, alpha=0.5, color='r')

In [ ]:
data_electronOnly = data[['ratio_mu0', 'ratio_mu1', 'ratio_mu2']]

data_max = data_electronOnly.sort_values(['ratio_mu0', 'ratio_mu1', 'ratio_mu2'])
max_ele0, max_ele1, max_ele2 = data_max.max()

max_common_ele0, max_common_ele1, max_common_ele2 = data_max.iloc[0]

figure = corner.corner(data_electronOnly.to_numpy())
corner.overplot_lines(figure, [max_ele0, max_ele1, max_ele2], lw=5, alpha=0.5)
corner.overplot_lines(figure, [max_common_ele0, max_common_ele1, max_common_ele2], lw=5, alpha=0.5, color='r')

In [ ]:
def plot_corner_byPlane(plane=''):
    max_all = data[f'ratio_total{plane}'].max()
    dataToPlot = data[[f'ratio_ele{plane}', f'ratio_gamma{plane}', f'ratio_mu{plane}', f'ratio_p{plane}', f'ratio_pi{plane}', ]]

    emax, gmax, mumax, pmax, pimax = dataToPlot.max()

    print(emax, gmax, mumax, pmax, pimax)
    
    fig = corner.corner(dataToPlot.to_numpy(), labels=['e', '$\\gamma$', '$\\mu$', 'p', '$\\pi$'], bins=10)
    corner.overplot_lines(fig, [emax, gmax, mumax, pmax, pimax], color='r', lw=5)
    corner.overplot_lines(fig, [max_all]*5)

In [ ]:
plot_corner_byPlane()

In [ ]:
plot_corner_byPlane(0)

In [ ]:
plot_corner_byPlane(1)

In [ ]:
plot_corner_byPlane(2)

In [ ]:
def plot_corner_byPlane(plane='', compareTo='ratio_total{}'):
    dataToPlot = data[[
        # f'ratio_ele{plane}', f'ratio_gamma{plane}', f'ratio_mu{plane}', f'ratio_p{plane}', f'ratio_pi{plane}', 
        'ratio_total{}'.format(plane), 
        f'roiThreshold_{plane}',
        # f'minPulseHeight_{plane}',
        # f'minPulseSigma_{plane}',
        f'LongMaxHits_{plane}',
        f'LongPulseWidth_{plane}',
        f'PulseHeightCuts_{plane}',
        # f'PulseWidthCuts_{plane}',
        # f'PulseRatioCuts_{plane}',
        'MaxMultiHit',
        'Chi2NDF',
    ]]
    
    fig = corner.corner(dataToPlot.to_numpy(), labels=[
        'Hit energy/IDE', 
        'ROI trh', 
        # 'MP height', 
        # 'MP $\\sigma$', 
        'LongMaxHits', 
        'LongPulseWidth', 
        'PulseHeightCuts', 
        # 'PulseWidthCuts', 
        # 'PulseRatioCuts', 
        'MaxMultiHit', 
        'Chi2NDF'
    ], bins=[15, 6, 4, 3, 2, 4, 5])
    return fig

In [ ]:
for plane in [0, 1, 2]:
    fig = plot_corner_byPlane(plane, compareTo='ratio_total{}')
    fig.savefig(f'ratio_total{plane}.png')
    for particle in ['ele', 'mu', 'p', 'pi']:
        fig = plot_corner_byPlane(plane, compareTo=f'ratio_{particle}{{}}')
        # fig.savefig(f'ratio_{particle}{plane}.png')

# New Metric 

We introduced a new metric. This allows all the values to be actually passed to the final `db` and more infor in the gridsearch stage

In [4]:
connection = sql.connect('/exp/icarus/data/users/msotgia/hitTuning/resultsToMerge/hitTuning_merged_resultsNewMetricMar22ndFlux.db')
data = pd.read_sql('SELECT * FROM runs', connection)

In [5]:
averageLabels = [
    'roiThreshold_0', 'roiThreshold_1', 'roiThreshold_2', 
    'minPulseHeight_0', 'minPulseHeight_1', 'minPulseHeight_2', 
    'minPulseSigma_0', 'minPulseSigma_1', 'minPulseSigma_2',
    'LongMaxHits_0', 'LongMaxHits_1', 'LongMaxHits_2', 
    'LongPulseWidth_0', 'LongPulseWidth_1', 'LongPulseWidth_2', 
    'PulseHeightCuts_0', 'PulseHeightCuts_1', 'PulseHeightCuts_2', 
    'PulseWidthCuts_0', 'PulseWidthCuts_1', 'PulseWidthCuts_2', 'PulseRatioCuts_0',
    'PulseRatioCuts_1', 'PulseRatioCuts_2', 
    'MaxMultiHit', 'Chi2NDF'
]

sumLabel = [
    'hitEnergy', 'ideEnergy', 
    'plane0HitEnergy', 'plane1HitEnergy', 'plane2HitEnergy',
    'plane0IdeEnergy', 'plane1IdeEnergy', 'plane2IdeEnergy'
]

filterLabels = ['protons', 'chargedPions', 'muons', 'electrons', 'neutralPions', 'gammas']
averageLabels2 = ['eventRatio', 'plane0Ratio', 'plane1Ratio', 'plane2Ratio']

aggregate = {col: 'mean' for col in averageLabels}
aggregate.update({col: 'mean' for col in averageLabels2})
aggregate.update({col: 'sum' for col in sumLabel})

In [20]:
def plot_for_plane(data_selected: pd.DataFrame, plane, selection_label, sample_label = f'Overlay + BNB $\\nu$-only (YZ sim. + 2D)', file_tag = 'BNBFlux_noPreselection'):
    dataToPlot = data_selected[['eventRatio', f'plane{plane}Ratio', f'roiThreshold_{plane}', f'LongMaxHits_{plane}', f'LongPulseWidth_{plane}', f'PulseHeightCuts_{plane}', 'MaxMultiHit', 'Chi2NDF']]
    dataToPlot = dataToPlot[(dataToPlot.eventRatio > 0) & (dataToPlot[f'plane{plane}Ratio'] > 0) & (dataToPlot[f'plane{plane}Ratio'] < 2)]

    keys = {
        f'roiThreshold_{plane}': 'ROI Threshold', 
        f'LongMaxHits_{plane}': 'LongMaxHits', 
        f'LongPulseWidth_{plane}': 'LongPulseWidth', 
        f'PulseHeightCuts_{plane}': 'PulseHeightCuts',
        'MaxMultiHit': 'MaxMultiHit', 
        'Chi2NDF': 'Chi2NDF'
    }

    planeMap = {
        0: 'Induction 1',
        1: 'Induction 2',
        2: 'Collection'
    }

    dataForRatio = data_selected.groupby('jobNum').agg(aggregate)
    dataForRatio['ratioPerJob'] = dataForRatio.hitEnergy / dataForRatio.ideEnergy
    dataForRatio[f'ratioPerJobPlane{plane}'] = dataForRatio[f'plane{plane}HitEnergy'] / dataForRatio[f'plane{plane}IdeEnergy']
    # print(dataForRatio.keys())
    dataForRatio = dataForRatio[[
        'eventRatio', 'ratioPerJob',
        f'plane{plane}Ratio', f'ratioPerJobPlane{plane}', 
        f'roiThreshold_{plane}', f'LongMaxHits_{plane}', f'LongPulseWidth_{plane}', f'PulseHeightCuts_{plane}', 'MaxMultiHit', 'Chi2NDF'
    ]].sort_values(f'ratioPerJobPlane{plane}')

    print('===============')
    print(f'For plane {planeMap[plane]}, with selection {file_tag}, the best values are')
    print(dataForRatio.sort_values(f"ratioPerJobPlane{plane}", key=lambda x: np.abs(x - 1)).head()[[
        'ratioPerJob', f'ratioPerJobPlane{plane}', f'roiThreshold_{plane}', f'LongMaxHits_{plane}', f'LongPulseWidth_{plane}', f'PulseHeightCuts_{plane}', 'MaxMultiHit', 'Chi2NDF'
    ]].T)
    print('===============\n')
    
    dataForRatio = dataForRatio[(dataForRatio['ratioPerJob'] > 0) & (dataForRatio[f'ratioPerJobPlane{plane}'] > 0) & (dataForRatio[f'ratioPerJobPlane{plane}'] < 2)]

    fig, axs = plt.subplots(ncols=len(keys), sharey=True, figsize=(16.5, 5))

    axs[0].set_title(sample_label, loc='left')
    axs[-1].set_title(f'{planeMap[plane]} / {selection_label}', loc='right')

    axs[0].set_ylabel(f'Hit energy/IDE')
    
    for key, ax in zip(keys, axs):
        ax.set_xlabel(f'{keys[key]}')
        ax.hist2d(np.array(dataToPlot[key]), np.array(dataToPlot[f'plane{plane}Ratio']), bins=[dataToPlot[key].unique().shape[0], 35]) #, ax=ax)
        ax.scatter(np.array(dataForRatio[key]), np.array(dataForRatio[f'plane{plane}Ratio']), marker='v', color='none', edgecolor='crimson', 
                   label=f'Avg. ratio (over events)\nOnly {planeMap[plane]}')
        ax.scatter(np.array(dataForRatio[key]), np.array(dataForRatio[f'ratioPerJobPlane{plane}']), marker='^', color='none', edgecolor='indianred', 
                   label=r'$\left(\sum_\mathrm{events} \sum_\mathrm{hits} E_\mathrm{hit}\right)/\left(\sum_\mathrm{events} \sum_\mathrm{IDEs} E_\mathrm{IDE}\right)$ (by plane)')
        ax.scatter(np.array(dataForRatio[key]), np.array(dataForRatio[f'ratioPerJob']), marker='o', color='none', edgecolor='red', 
                   label=r'$\left(\sum_\mathrm{events} \sum_\mathrm{hits} E_\mathrm{hit}\right)/\left(\sum_\mathrm{events} \sum_\mathrm{IDEs} E_\mathrm{IDE}\right)$')
        # corner.hist2d(np.array(dataToPlot[key]), np.array(dataToPlot[f'plane{plane}Ratio']), bins=[dataToPlot[key].unique().shape[0], 35], ax=ax, plot_density=False)

    axs[0].legend(fancybox=False, loc='lower left')
    fig.tight_layout()
    fig.savefig(f'plots/fromGridSearch/searchingGrid_{file_tag}_plane{plane}.pdf', bbox_inches='tight')

    fig, ax = plt.subplots(figsize=(7, 7))
    labels = ['Ratio (by event, avg.)', 'Ratio', 'Ratio (avg, by plane)', 'Ratio (by plane)', 'ROI Threshold', 'LongMaxHits', 'LongPulseWidth', 'PulseHeightCuts', 'MaxMultiHit', 'Chi2NDF']
    hm = sns.heatmap(
        dataForRatio.corr(), 
        annot=True, 
        cmap="coolwarm", 
        vmin=-1, vmax=1, 
        xticklabels=labels,
        yticklabels=labels,
        square=True,
        annot_kws={"size": 8},
        ax=ax
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_color("black")
    
    cbar = hm.collections[0].colorbar
    for spine in cbar.ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_color("black")

    ax.set_title(f'{sample_label}\n{planeMap[plane]} / {selection_label}', loc='left')

    fig.savefig(f'plots/fromGridSearch/searchingGrid_{file_tag}_plane{plane}_correlation.pdf', bbox_inches='tight')

In [22]:
data1MuNpNpi = data[
    (data.electrons == 0)
  & (data.gammas == 0)
  & (data.neutralPions == 0)
]

for plane in range(3):
    plot_for_plane(data1MuNpNpi, plane, '$1\\mu \\mathrm{NpN}\\pi$ events', file_tag = 'BNBFlux_selecting1MuNpMPiEvents')

For plane Induction 1, with selection BNBFlux_selecting1MuNpMPiEvents, the best values are
jobNum                    1651         3008        3005         3007  \
ratioPerJob           1.067437     1.067438    1.067438     1.067438   
ratioPerJobPlane0     1.065484     1.065486    1.065486     1.065486   
roiThreshold_0        4.000000     2.000000    2.000000     2.000000   
LongMaxHits_0        10.000000    10.000000   10.000000    10.000000   
LongPulseWidth_0      5.000000    10.000000   10.000000    10.000000   
PulseHeightCuts_0     2.000000     2.000000    2.000000     2.000000   
MaxMultiHit          10.000000     7.000000    7.000000     7.000000   
Chi2NDF            1000.000000  2000.000000  500.000000  1500.000000   

jobNum                    3001  
ratioPerJob           1.067438  
ratioPerJobPlane0     1.065486  
roiThreshold_0        2.000000  
LongMaxHits_0        10.000000  
LongPulseWidth_0     10.000000  
PulseHeightCuts_0     2.000000  
MaxMultiHit           5.00000

In [23]:
data1MuNp0pi = data[
    (data.electrons == 0)
  & (data.gammas == 0)
  & (data.neutralPions == 0)
  & (data.chargedPions == 0)
]

for plane in range(3):
    plot_for_plane(data1MuNp0pi, plane, '$1\\mu \\mathrm{Np} 0\\pi$ events', file_tag = 'BNBFlux_selecting1MuNp0PiEvents')

For plane Induction 1, with selection BNBFlux_selecting1MuNp0PiEvents, the best values are
jobNum                    1651         3008        3005         3007  \
ratioPerJob           1.067437     1.067438    1.067438     1.067438   
ratioPerJobPlane0     1.065484     1.065486    1.065486     1.065486   
roiThreshold_0        4.000000     2.000000    2.000000     2.000000   
LongMaxHits_0        10.000000    10.000000   10.000000    10.000000   
LongPulseWidth_0      5.000000    10.000000   10.000000    10.000000   
PulseHeightCuts_0     2.000000     2.000000    2.000000     2.000000   
MaxMultiHit          10.000000     7.000000    7.000000     7.000000   
Chi2NDF            1000.000000  2000.000000  500.000000  1500.000000   

jobNum                    3001  
ratioPerJob           1.067438  
ratioPerJobPlane0     1.065486  
roiThreshold_0        2.000000  
LongMaxHits_0        10.000000  
LongPulseWidth_0     10.000000  
PulseHeightCuts_0     2.000000  
MaxMultiHit           5.00000

### Wit the BNB $\nu_\mathrm{e}$ sample

In [24]:
connectionNuEs = sql.connect('/exp/icarus/data/users/msotgia/hitTuning/resultsToMerge/hitTuning_merged_resultsNewMetricMar22ndNues.db')
dataNuEs = pd.read_sql('SELECT * FROM runs', connectionNuEs)

In [25]:
data1eNPNGammaNpi0 = dataNuEs[
    (dataNuEs.muons == 0)
  & (dataNuEs.chargedPions == 0)
]

for plane in range(3):
    plot_for_plane(data1eNPNGammaNpi0, plane, '$1\\mathrm{e Np N}\\pi^{0}$ events', 
                   sample_label = f'Overlay + BNB $\\nu_\\mathrm{{e}}$-only (YZ sim. + 2D)', file_tag = 'BNBFlux_selecting1eNPNGammaNpi0')



For plane Induction 1, with selection BNBFlux_selecting1eNPNGammaNpi0, the best values are
jobNum                   3390        3405        3385        3380        3400
ratioPerJob          0.742216    0.742216    0.742216    0.742216    0.742216
ratioPerJobPlane0    0.737985    0.737985    0.737985    0.737985    0.737985
roiThreshold_0       1.000000    1.000000    1.000000    1.000000    1.000000
LongMaxHits_0        5.000000    5.000000    5.000000    5.000000    5.000000
LongPulseWidth_0     2.000000    5.000000    2.000000    2.000000    5.000000
PulseHeightCuts_0    3.000000    2.000000    3.000000    3.000000    2.000000
MaxMultiHit         10.000000    7.000000    7.000000    5.000000    5.000000
Chi2NDF            500.000000  500.000000  500.000000  500.000000  500.000000

For plane Induction 2, with selection BNBFlux_selecting1eNPNGammaNpi0, the best values are
jobNum                   3370        3395        3365        3385        3380
ratioPerJob          0.742216    0.74

In [26]:
data1eNPNGammaNpi0 = dataNuEs[
    (dataNuEs.muons == 0)
  & (dataNuEs.chargedPions == 0)
  & (dataNuEs.neutralPions == 0)
]

for plane in range(3):
    plot_for_plane(data1eNPNGammaNpi0, plane, '$1\\mathrm{e Np} 0\\pi^{0}$ events', 
                   sample_label = f'Overlay + BNB $\\nu_\\mathrm{{e}}$-only (YZ sim. + 2D)', file_tag = 'BNBFlux_selecting1eNPNGammaNpi0')

For plane Induction 1, with selection BNBFlux_selecting1eNPNGammaNpi0, the best values are
jobNum                   3380        3365        3400        3405        3370
ratioPerJob          0.730944    0.730944    0.730944    0.730944    0.730944
ratioPerJobPlane0    0.725409    0.725409    0.725409    0.725409    0.725409
roiThreshold_0       1.000000    1.000000    1.000000    1.000000    1.000000
LongMaxHits_0        5.000000    5.000000    5.000000    5.000000    5.000000
LongPulseWidth_0     2.000000    2.000000    5.000000    5.000000    2.000000
PulseHeightCuts_0    3.000000    2.000000    2.000000    2.000000    2.000000
MaxMultiHit          5.000000    7.000000    5.000000    7.000000   10.000000
Chi2NDF            500.000000  500.000000  500.000000  500.000000  500.000000

For plane Induction 2, with selection BNBFlux_selecting1eNPNGammaNpi0, the best values are
jobNum                   3405        3370        3380        3390        3385
ratioPerJob          0.730944    0.73

In [37]:
for key in data.keys():
    uniques = data[key].unique()
    if len(uniques) < 10:
        print(f'Data key "{key}" has unique values {uniques}')

Data key "roiThreshold_0" has unique values [3. 6. 5. 4. 1. 2.]
Data key "roiThreshold_1" has unique values [3. 6. 5. 4. 1. 2.]
Data key "roiThreshold_2" has unique values [3. 6. 5. 4. 1. 2.]
Data key "minPulseHeight_0" has unique values [2.]
Data key "minPulseHeight_1" has unique values [2.]
Data key "minPulseHeight_2" has unique values [2.]
Data key "minPulseSigma_0" has unique values [1.]
Data key "minPulseSigma_1" has unique values [1.]
Data key "minPulseSigma_2" has unique values [1.]
Data key "LongMaxHits_0" has unique values [ 1 10 15  5]
Data key "LongMaxHits_1" has unique values [ 1 10 15  5]
Data key "LongMaxHits_2" has unique values [ 1 10 15  5]
Data key "LongPulseWidth_0" has unique values [ 5.  8.  2. 10.]
Data key "LongPulseWidth_1" has unique values [ 5.  8.  2. 10.]
Data key "LongPulseWidth_2" has unique values [ 5.  8.  2. 10.]
Data key "PulseHeightCuts_0" has unique values [3. 2.]
Data key "PulseHeightCuts_1" has unique values [3. 2.]
Data key "PulseHeightCuts_2" has